In [2]:
import os, sys
import pandas as pd

res = {'acd': {'bert_clf': {},
               'llm': {}},
       'acsa': {'bert_clf': {},
                'hier_gcn': {},
                'llm': {}},
       'tasd': {'dlo': {},
                'llm': {}}}

METHOD = 'llm'
TASK = 'tasd'

TASK_METHOD = [['acd', 'bert_clf'], 
               ['acd', 'llm'],
               ['acsa', 'bert_clf'],
               ['acsa', 'hier_gcn'],
               ['acsa', 'llm'],
               ['tasd', 'dlo'],
               ['tasd', 'llm']
              ]

for TASK, METHOD in TASK_METHOD:

    col_names = ['task', 'method', 'lang', 'lang_setting', 'eval_type', 'data_setting', 'learning_rate', 'batch_size', 'epoch', 'seed', 'f1-micro', 'f1-macro', 'accuracy']
    config_cols = ["lang", "learning_rate", "epoch"]
    RESULTS_PATH = f'../results/{METHOD}/'
    folder_names = [folder for folder in os.listdir(os.path.join(RESULTS_PATH)) if os.path.isdir(os.path.join(RESULTS_PATH, folder)) and folder not in ['.ipynb_checkpoints', 'checkpoints']]
    runs = []
    for folder_name in folder_names:
        try:
            cond_parameters = folder_name.split('_')
            if cond_parameters[0] == 'acd':
                df = pd.read_csv(os.path.join(RESULTS_PATH, folder_name, 'metrics_asp.tsv'), sep = '\t')
            elif cond_parameters[0] == 'acsa':
                df = pd.read_csv(os.path.join(RESULTS_PATH, folder_name, 'metrics_asp_pol.tsv'), sep = '\t')
            elif cond_parameters[0] == 'tasd':
                df = pd.read_csv(os.path.join(RESULTS_PATH, folder_name, 'metrics_phrases.tsv'), sep = '\t')
                
            df = df.set_index(df.columns[0])
            
            cond_parameters = folder_name.split('_')
            
            if cond_parameters[3] == 'eval':
                cond_parameters[3] = cond_parameters[3] + '_' + cond_parameters[4]
                cond_parameters.pop(4)
        
            if cond_parameters[4] == 'multi':
                cond_parameters[4] = cond_parameters[4] + '-' + cond_parameters[5]
                cond_parameters.pop(5)
            
            cond_parameters[1:1] = [METHOD]
                
            cond_parameters.append(df.loc['Micro-AVG', 'f1'])
            cond_parameters.append(df.loc['Macro-AVG', 'f1'])
            cond_parameters.append(df.loc['Micro-AVG', 'accuracy'])
            runs.append(cond_parameters)
        except:
            print(folder_name)
            pass
    
    results_all = pd.DataFrame(runs, columns = col_names)
    results_all["f1-micro"] = pd.to_numeric(results_all["f1-micro"], errors="coerce")
    results_all["seed"] = results_all["seed"].astype(int)
    
    if METHOD != 'llm':
        
        orig = results_all[ \
        (results_all['method'] == METHOD) 
        & (results_all['eval_type'] == 'test') 
        & (results_all['task'] == TASK) 
        & (results_all['data_setting'] == 'orig-o') 
        & (results_all['lang_setting'] == 'adapted') 
        ].drop_duplicates()
        
        adapted = results_all[ \
        (results_all['method'] == METHOD) 
        & (results_all['eval_type'] == 'test') 
        & (results_all['task'] == TASK) 
        & (results_all['data_setting'] == 'balanced-o') 
        & (results_all['lang_setting'] == 'adapted') 
        ].drop_duplicates()
        
        balanced = results_all[ \
        (results_all['method'] == METHOD) 
        & (results_all['eval_type'] == 'test') 
        & (results_all['task'] == TASK) 
        & (results_all['data_setting'] == 'balanced-o') 
        & (results_all['lang_setting'] == 'orig') 
        ].drop_duplicates()
    
        # Gruppiere und filtere auf Gruppen mit genau eval_count Einträgen
        
        adapted_grp = adapted.groupby(config_cols).filter(lambda x: len(x) == 5)
        # Berechne dann den Durchschnitt nur über diese Gruppen
        adapted_best = adapted_grp.groupby(config_cols)[["f1-micro", "f1-macro"]].mean()
    
    else:
        orig = results_all[ \
        (results_all['method'] == METHOD) 
        & (results_all['eval_type'] == 'test') 
        & (results_all['task'] == TASK) 
        & (results_all['data_setting'] == 'orig-o') 
        & (results_all['lang_setting'] == 'orig') 
        ].drop_duplicates()
        
        adapted_best = pd.DataFrame()
    
        balanced = results_all[ \
        (results_all['method'] == METHOD) 
        & (results_all['eval_type'] == 'test') 
        & (results_all['task'] == TASK) 
        & (results_all['data_setting'] == 'balanced-o') 
        & (results_all['lang_setting'] == 'orig') 
        ].drop_duplicates()
    
    orig_grp = orig.groupby(config_cols).filter(lambda x: len(x) == 5)
    balanced_grp = balanced.groupby(config_cols).filter(lambda x: len(x) == 5)
    
    orig_best = orig_grp.groupby(config_cols)[["f1-micro", "f1-macro"]].mean()
    balanced_best = balanced_grp.groupby(config_cols)[["f1-micro", "f1-macro"]].mean()

    LANGS = ['cs', 'de', 'en', 'es', 'fr', 'nl', 'ru']
    res[TASK][METHOD] = {'orig': {key: '-' for key in LANGS}, 
                             'balanced': {key: '-' for key in LANGS}, 
                             'adapted': {key: '-' for key in LANGS}}
    
    for LANG in LANGS:
        res[TASK][METHOD]['orig'][LANG] = round(orig_best.xs(LANG, level="lang")["f1-micro"].values[0]*100,2)
    
        res[TASK][METHOD]['balanced'][LANG] = round(balanced_best.xs(LANG, level="lang")["f1-micro"].values[0]*100,2)
    
        try:
            res[TASK][METHOD]['adapted'][LANG] = round(adapted_best.xs(LANG, level="lang")["f1-micro"].values[0]*100,2)
        except:
            res[TASK][METHOD]['adapted'][LANG] = '--'
        
# c = ''
# order  = ['cs', 'de', 'en', 'es', 'fr', 'nl', 'ru']
# for b in order:
#     for df in [balanced_best, adapted_best, orig_best]:
#         try:
#             c += '& ' + f'{df.xs(b, level="lang")["f1-micro"].values[0]*100:.2f} '
#         except:
#             c += '& -- '

# c += ' \\'
# c

tasd_cs_orig_eval_0_balanced-o_0.0002_16_10_5
tasd_fr_orig_eval_0_balanced-o_0.0002_16_15_5
tasd_nl_orig_eval_0_balanced-o_0.0002_16_13_5
tasd_nl_orig_eval_0_balanced-o_0.0002_16_8_5
tasd_cs_orig_eval_0_balanced-o_0.0002_16_15_5
tasd_es_orig_eval_0_balanced-o_0.0002_16_13_5
tasd_de_adapted_eval_0_balanced-o_0.0002_16_5_5
tasd_cs_orig_eval_0_balanced-o_0.0002_16_5_5
tasd_fr_orig_eval_0_balanced-o_0.0002_16_13_5
tasd_es_orig_eval_0_balanced-o_0.0002_16_15_5
tasd_ru_orig_eval_0_balanced-o_0.0002_16_8_5
tasd_es_orig_eval_0_balanced-o_0.0002_16_10_5
tasd_ru_orig_eval_0_balanced-o_0.0002_16_10_5
tasd_cs_orig_eval_0_balanced-o_0.0002_16_13_5
tasd_fr_orig_eval_0_balanced-o_0.0002_16_10_5
tasd_fr_orig_eval_0_balanced-o_0.0002_16_8_5
tasd_cs_orig_eval_0_balanced-o_0.0002_16_8_5
acd_de_orig_eval_0_balanced-b_0.0002_16_20_5
tasd_ru_orig_eval_0_balanced-o_0.0002_16_13_5
tasd_es_orig_eval_0_balanced-o_0.0002_16_8_5
tasd_nl_orig_eval_0_balanced-o_0.0002_16_15_5
tasd_nl_orig_eval_0_balanced-o_0.0002_1